# Modul 12: Eigene Orchestrierung — Patterns & Implementierung

**Agent Systems: Vom Konzept zum eigenen Orchestrator** | 🔴 Developer Track

In diesem Notebook implementierst du Task-Dekomposition, Sub-Task-Spawning und Result-Aggregation.

🎯 **Lernziel:** Das Orchestrator-Worker-Pattern implementieren: Aufgabe zerlegen → parallel bearbeiten → Ergebnisse zusammenführen.

---

## Planning-Pattern (Ng, 2024 — Pattern 3/4)

```
  KOMPLEXE AUFGABE
       │
       ▼
  ┌──────────────┐
  │  DECOMPOSE   │  ← Aufgabe in Sub-Tasks zerlegen
  └──────┬───────┘
    ┌────┼────┐
    ▼    ▼    ▼
  [T1] [T2] [T3]   ← Parallel oder sequentiell
    │    │    │
    └────┼────┘
         ▼
  ┌──────────────┐
  │  AGGREGATE   │  ← Ergebnisse zusammenführen
  └──────────────┘
```

**Patterns:** Sequential (A→B→C), Concurrent (A∥B∥C→Merge), Handoff (A→B), Orchestrator-Worker

In [ ]:
# Task-Dekomposition + Sub-Task-Spawning

import time
from datetime import datetime


class SubTask:
    """Ein einzelner Sub-Task."""
    def __init__(self, task_id, description, worker_type='general'):
        self.task_id = task_id
        self.description = description
        self.worker_type = worker_type
        self.status = 'pending'
        self.result = None
        self.duration = 0

    def __repr__(self):
        icon = {'pending': '⏳', 'running': '⚙️', 'completed': '✅', 'failed': '❌'}[self.status]
        return f'{icon} [{self.task_id}] {self.description} ({self.status})'


class TaskDecomposer:
    """Zerlegt komplexe Aufgaben in Sub-Tasks."""

    # Keyword-basierte Zerlegungsregeln
    RULES = {
        'recherchiere': {'type': 'research', 'subtasks': [
            ('Quellen sammeln', 'research'),
            ('Quellen bewerten', 'review'),
            ('Zusammenfassung erstellen', 'writing'),
        ]},
        'vergleiche': {'type': 'comparison', 'subtasks': [
            ('Option A analysieren', 'research'),
            ('Option B analysieren', 'research'),
            ('Vergleichstabelle erstellen', 'writing'),
        ]},
        'erstelle': {'type': 'creation', 'subtasks': [
            ('Struktur planen', 'planning'),
            ('Inhalt generieren', 'writing'),
            ('Qualitaetscheck', 'review'),
        ]},
    }

    def decompose(self, task):
        """Zerlegt eine Aufgabe in Sub-Tasks."""
        task_lower = task.lower()
        subtasks = []

        for keyword, config in self.RULES.items():
            if keyword in task_lower:
                print(f'\n  Erkannt: "{keyword}" → {config["type"]}-Pattern')
                for i, (desc, worker) in enumerate(config['subtasks']):
                    st = SubTask(
                        task_id=f'T{i+1}',
                        description=f'{desc} ({task})',
                        worker_type=worker
                    )
                    subtasks.append(st)
                return subtasks

        # Fallback: Aufgabe nicht zerlegbar
        subtasks.append(SubTask('T1', task, 'general'))
        return subtasks


class SubTaskRunner:
    """Simuliert die Ausfuehrung von Sub-Tasks."""

    WORKERS = {
        'research': lambda desc: f'3 Quellen gefunden zu: {desc[:40]}...',
        'review': lambda desc: f'Qualitaet geprueft: 8/10 fuer: {desc[:40]}...',
        'writing': lambda desc: f'Text erstellt (450 Woerter) fuer: {desc[:40]}...',
        'planning': lambda desc: f'Struktur: 3 Abschnitte geplant fuer: {desc[:40]}...',
        'general': lambda desc: f'Verarbeitet: {desc[:50]}...',
    }

    def run(self, subtask):
        """Fuehrt einen Sub-Task aus."""
        subtask.status = 'running'
        worker = self.WORKERS.get(subtask.worker_type, self.WORKERS['general'])
        try:
            subtask.result = worker(subtask.description)
            subtask.status = 'completed'
        except Exception as e:
            subtask.result = str(e)
            subtask.status = 'failed'
        return subtask


class ResultAggregator:
    """Aggregiert Ergebnisse von Sub-Tasks."""

    def aggregate(self, subtasks, strategy='sequential'):
        """Fasst alle Ergebnisse zusammen."""
        completed = [st for st in subtasks if st.status == 'completed']
        failed = [st for st in subtasks if st.status == 'failed']

        print(f'\n📊 Aggregation ({strategy}):')
        print(f'  Abgeschlossen: {len(completed)}/{len(subtasks)}')
        if failed:
            print(f'  Fehlgeschlagen: {len(failed)}')

        results = []
        for st in completed:
            results.append(st.result)
            print(f'  {st.task_id}: {st.result}')

        # Synthese
        synthesis = f'Zusammenfassung aus {len(completed)} Ergebnissen: ' + ' | '.join(
            r[:30] + '...' for r in results
        )
        print(f'\n  💡 Synthese: {synthesis}')
        return synthesis


# === Demo: Vollstaendige Pipeline ===
print('=== Orchestrator Pipeline ===\n')

decomposer = TaskDecomposer()
runner = SubTaskRunner()
aggregator = ResultAggregator()

# Komplexe Aufgabe
task = 'Recherchiere zum Thema Agent-Architekturen 2025'

# 1. Dekomposition
print(f'Aufgabe: "{task}"')
subtasks = decomposer.decompose(task)
print(f'\n📋 {len(subtasks)} Sub-Tasks erstellt:')
for st in subtasks:
    print(f'  {st}')

# 2. Ausfuehrung
print('\n⚙️ Ausfuehrung:')
for st in subtasks:
    runner.run(st)
    print(f'  {st}')

# 3. Aggregation
aggregator.aggregate(subtasks)

In [ ]:
# Verschiedene Orchestration-Patterns

def run_sequential(subtasks, runner):
    """Sequential: A → B → C (Ergebnis von A fliesst in B)"""
    print('\n=== Sequential Pattern ===')
    previous_result = None
    for st in subtasks:
        if previous_result:
            st.description += f' (basierend auf: {previous_result[:30]})'
        runner.run(st)
        previous_result = st.result
        print(f'  {st}')
    return subtasks


def run_concurrent(subtasks, runner):
    """Concurrent: A ∥ B ∥ C → Merge (alle parallel, dann zusammen)"""
    print('\n=== Concurrent Pattern ===')
    # Simuliert parallele Ausfuehrung
    for st in subtasks:
        runner.run(st)
        print(f'  {st}')
    print('  → Alle parallel abgeschlossen')
    return subtasks


def run_with_error_handling(subtasks, runner, max_retries=2):
    """Mit Fehlerbehandlung: Retry bei Fehler."""
    print('\n=== Pattern mit Error-Handling ===')
    for st in subtasks:
        for attempt in range(max_retries + 1):
            runner.run(st)
            if st.status == 'completed':
                print(f'  {st}')
                break
            elif attempt < max_retries:
                print(f'  ⚠️ Retry {attempt + 1} fuer {st.task_id}...')
                st.status = 'pending'
            else:
                print(f'  ❌ {st.task_id} endgueltig fehlgeschlagen nach {max_retries} Retries')
    return subtasks


# Demo: Vergleich der Patterns
task2 = 'Vergleiche OpenClaw vs LangGraph'
subtasks2 = decomposer.decompose(task2)

print(f'\nAufgabe: "{task2}"')
print(f'{len(subtasks2)} Sub-Tasks: {[st.task_id for st in subtasks2]}')

# Sequential
seq_tasks = [SubTask(f'S{i+1}', st.description, st.worker_type) for i, st in enumerate(subtasks2)]
run_sequential(seq_tasks, runner)
aggregator.aggregate(seq_tasks, 'sequential')

# Concurrent
con_tasks = [SubTask(f'C{i+1}', st.description, st.worker_type) for i, st in enumerate(subtasks2)]
run_concurrent(con_tasks, runner)
aggregator.aggregate(con_tasks, 'concurrent')

In [ ]:
# 🎯 ÜBUNG: Baue eine Custom-Dekomposition
#
# Aufgabe:
# 1. Fuege eine neue Zerlegungsregel fuer 'analysiere' hinzu
#    Sub-Tasks: Daten sammeln (research), Muster erkennen (review), Bericht schreiben (writing)
# 2. Teste mit der Aufgabe: 'Analysiere die Kosten von AI APIs'
# 3. Fuehre die Sub-Tasks aus und aggregiere die Ergebnisse

# TODO: Neue Regel hinzufuegen
# decomposer.RULES['analysiere'] = {
#     'type': 'analysis',
#     'subtasks': [...]
# }

# TODO: Testen
# task3 = 'Analysiere die Kosten von AI APIs'
# subtasks3 = decomposer.decompose(task3)
# for st in subtasks3:
#     runner.run(st)
# aggregator.aggregate(subtasks3)

In [ ]:
# ✅ LÖSUNG

decomposer.RULES['analysiere'] = {
    'type': 'analysis',
    'subtasks': [
        ('Daten sammeln', 'research'),
        ('Muster erkennen', 'review'),
        ('Bericht schreiben', 'writing'),
    ]
}

task3 = 'Analysiere die Kosten von AI APIs'
print(f'Aufgabe: "{task3}"\n')

subtasks3 = decomposer.decompose(task3)
print(f'\n{len(subtasks3)} Sub-Tasks:')
for st in subtasks3:
    print(f'  {st}')

print('\nAusfuehrung:')
for st in subtasks3:
    runner.run(st)
    print(f'  {st}')

aggregator.aggregate(subtasks3, 'analysis-pipeline')

## Orchestration Patterns im Vergleich

| Pattern | Wann nutzen? | Fehlerverhalten |
|---------|-------------|------------------|
| Sequential | Jeder Schritt braucht vorherigen | Pipeline stoppt |
| Concurrent | Unabhängige Teilaufgaben | Teilweise Ergebnisse möglich |
| Handoff | Spezialist übernimmt | Fallback an Orchestrator |
| Orch-Worker | Komplexe Aufgaben | Retry + Partial Results |

---

### ✅ Self-Check
- [ ] Ich kann eine Aufgabe in Sub-Tasks zerlegen
- [ ] Ich verstehe den Unterschied zwischen Sequential und Concurrent
- [ ] Ich kann Ergebnisse von Sub-Tasks aggregieren

**→ Weiter zu [Modul 13: Production Readiness](https://colab.research.google.com/github/planck-lab/agent-systems/blob/main/notebooks/modul13_production.ipynb)**